### Workflow, um ein einfaches Spiel mit dem MCV-Pattern zu erstellen
1. Erstelle einen Prototyp der Spielklasse.
   Speichere diese dann ein einem File, z.b. `viergewinnt.py`.
2. Importiere dann die Spielklasse aus diesem File. Benuzte `importlib` um
   Änderungen im File ohne Kernelneustart reimportieren zu können.
3. Verfahre gleich beim Erstellen einer View-Klasse zur Darstellung.

In [ ]:
from model_view_controller import Observable, notify, BaseView, Controller


class VierGewinnt(Observable):
    def __init__(self):
        self.nrow = 6
        self.ncol = 7
        self.cols = [[] for _ in range(self.ncol)]

    @notify
    def new_game(self):
        self.result = None
        self.ptm = 0  # player to move
        for col in self.cols:
            col.clear()

    @notify
    def place_stone(self, col):
        player = self.ptm
        row = len(self.cols[col])

        self.cols[col].append(player)

        self.ptm = 1 - self.ptm
        return player, (col, row)

    def __repr__(self):
        return '\n'.join([f'{col}' for col in self.cols])

In [ ]:
game = VierGewinnt()
game.register_callback(print)

In [ ]:
game.new_game()
for col in (3, 4, 3, 4, 5, 6):
    game.place_stone(col)
game

In [ ]:
import importlib
import viergewinnt
importlib.reload(viergewinnt)
VierGewinnt = viergewinnt.VierGewinnt

In [ ]:
from model_view_controller import BaseView, Controller
from gridhelper import GridHelper


class View(BaseView):
    def __init__(self, game, width=180, height=160, nlayers=2, debug=True):
        super().__init__(game, width, height, nlayers, debug)

        self.colors = ['red', 'blue']
        self.bg, self.fg = self.mcanvas

        self.gridhelper = GridHelper(20, 20, 20, 20, 7,  6)
        self.gridhelper.draw_grid(self.fg, line_width=2, color='blue')

        self.log('Drawing the Grid')

    def update(self, event, data):
        self.log(f'running update(event={event}, data={data})')
        if event == 'new_game':
            self.bg.clear()
        if event == 'place_stone':
            player, (col, row) = data
            self.gridhelper.fill_circle(self.bg, (col, self.game.nrow - row - 1), color=self.colors[player])


game = VierGewinnt()
view = View(game)
view

In [ ]:
game.new_game()
game.place_stone(1)

In [ ]:
game.place_stone(1)

In [ ]:
game = VierGewinnt()

callbacks = {'n': game.new_game}
for i in range(1, 8):
    callbacks[str(i)] = lambda col=i: game.place_stone(col)


view = View(game)
controller = Controller(game, view, callbacks)
controller

In [ ]:
from model_view_controller import Controller

game = VierGewinnt()


def on_mouse_down(self, x, y, state):
    col, row = self.view.gridhelper.xy2cr(x, y)
    self.game.place_stone(col)


callbacks = {'n': game.new_game,
             'mouse_down': on_mouse_down,
             }


view = View(game)
controller = Controller(game, view, callbacks)
controller